In [ ]:
import os
import rarfile
import requests
import pandas as pd
from datetime import date

from bs4 import BeautifulSoup, Tag
from sina.config.credentials import (
    qqp_url,
    datos_abiertos_url, 
)

year = str(date.today().year)

In [35]:
page = requests.get(qqp_url)
soup = BeautifulSoup(page.content, 'lxml')

download_link: str | None = None

for link in soup.find_all('a'):
    if not isinstance(link, Tag):  
        continue
    if year in link.get_text():
        href = link.get('href')
        if href:
            download_link = str(href)
            break

if download_link:
    url = "/".join([
        datos_abiertos_url.rstrip("/"),
        download_link.lstrip("/")
    ])

    rar_response = requests.get(url)
    rar_response.raise_for_status()

    with open('temp.rar', 'wb') as f:
        f.write(rar_response.content)

    with rarfile.RarFile('temp.rar') as rf:
        csv_files = [
            info for info in rf.infolist()
            if info.filename.lower().endswith('.csv') and year in info.filename
        ]

        if not csv_files:
            raise FileNotFoundError(f"No se encontraron CSVs con el año {year} en el RAR")

        newest_csv_info = max(csv_files, key=lambda x: x.date_time)

        print(f"📦 CSV más reciente: {newest_csv_info.filename}")
        print(f"🕒 Modificado: {newest_csv_info.date_time}")

        with rf.open(newest_csv_info) as csv_file_in_rar:
            df_qqp = pd.read_csv(
                csv_file_in_rar,
                encoding='utf-8',
                header=None,
                low_memory=False
            )

os.remove('temp.rar')

📦 CSV más reciente: 2026/02-2026_Q2.csv
🕒 Modificado: (2026, 3, 9, 19, 54, 37)


In [36]:
nombres_columnas = [
    'PRODUCTO',
    'PRESENTACION',
    'MARCA',
    'CATEGORIA',
    'CATALOGO',
    'PRECIO',
    'FECHAREGISTRO',
    'CADENACOMERCIAL',
    'GIRO',
    'NOMBRECOMERCIAL', # Cambiado de 'NOMBRE_SUCURSAL' para coincidir con tu lista
    'DIRECCION',
    'ESTADO', # Movido para coincidir con el orden
    'MUNICIPIO', # Movido para coincidir con el orden
    'LATITUD',
    'LONGITUD'
]

df_qqp.columns = nombres_columnas

In [37]:
df_qqp[df_qqp['MUNICIPIO'] == 'Hermosillo'].head()

,PRODUCTO,PRESENTACION,MARCA,CATEGORIA,CATALOGO,PRECIO,FECHAREGISTRO,CADENACOMERCIAL,GIRO,NOMBRECOMERCIAL,DIRECCION,ESTADO,MUNICIPIO,LATITUD,LONGITUD
468235,Aceite,Botella 850 Ml. Vegetal,More Value,Aceites y Grasas Veg. Comestibles,Pacic,26.9,2026/02/16,Ley,Supermercado / Tienda de Autoservicio,Ley Sucursal Vado del Rio,Periferico Poniente. Esq. Vado del Rio. Col. P...,Sonora,Hermosillo,29.066052,-110.969103
468236,Arroz,Bolsa 900 Gr. Super Extra,Ley,Arroz y Cereales Preparados,Pacic,15.9,2026/02/16,Ley,Supermercado / Tienda de Autoservicio,Ley Sucursal Vado del Rio,Periferico Poniente. Esq. Vado del Rio. Col. P...,Sonora,Hermosillo,29.066052,-110.969103
468237,Atén,Lata 130 Gr. en Desmenuzado en Agua,Atén la Mar,Pescados y Mariscos en Conserva,Pacic,12.9,2026/02/16,Ley,Supermercado / Tienda de Autoservicio,Ley Sucursal Vado del Rio,Periferico Poniente. Esq. Vado del Rio. Col. P...,Sonora,Hermosillo,29.066052,-110.969103
468238,Azécar,Bolsa Plµstico 900 Gr. Estµndar,Ley,Azucar,Pacic,20,2026/02/16,Ley,Supermercado / Tienda de Autoservicio,Ley Sucursal Vado del Rio,Periferico Poniente. Esq. Vado del Rio. Col. P...,Sonora,Hermosillo,29.066052,-110.969103
468239,Carne Cerdo,1 Kg. Granel. Chuleta de Paleta de Puerco,S/M,Carne y Visceras de Cerdo,Pacic,61.5,2026/02/16,Ley,Supermercado / Tienda de Autoservicio,Ley Sucursal Vado del Rio,Periferico Poniente. Esq. Vado del Rio. Col. P...,Sonora,Hermosillo,29.066052,-110.969103


In [44]:
df_qqp_hmo = df_qqp[df_qqp['MUNICIPIO'] == 'Hermosillo'].copy()

In [45]:
df_qqp_hmo.PRODUCTO.unique()

array(['Aceite', 'Arroz', 'Atén', 'Azécar', 'Carne Cerdo', 'Carne Pollo',
       'Carne Res', 'Cebolla', 'Chile Fresco', 'Frijol', 'Huevo',
       'Jabàn de Tocador', 'Jitomate', 'Leche Ultrapasteurizada', 'Limàn',
       'Manzana', 'Pan de Caja', 'Papa', 'Papel Higi\x90nico',
       'Pasta para Sopa', 'Plµtano', 'Sardina', 'Tortilla de Maöz',
       'Zanahoria', 'Celulares', 'Componentes de Audio',
       'Electrànicos de Video', 'Estufas',
       'Extractores de Jugos y Exprimidores', 'Horno de Microondas',
       'Lavadoras', 'Licuadoras', 'Pantallas', 'Planchas',
       'Refrigeradores', 'Ventiladores', 'Acondicionador y Enjuague',
       'Agua con Gas', 'Agua Sin Gas', 'Aguacate', 'Ajo', 'Alcohol',
       'Algodàn', 'Alimento Preparado para Ni¥os', 'Alimentos Preparados',
       'Alubia', 'Apio', 'Avena', 'Bebidas Hidratantes', 'Betabel',
       'Bolögrafo Tinta Negra', 'Brandy', 'Bràcoli', 'Cacahuates',
       'Caf\x90 Soluble', 'Caf\x90 Tostado y Molido', 'Cafeteras',
       'Ca

In [49]:
CHAR_MAP = {
    'é': 'ú',     # Azécar → Azúcar
    'à': 'ó',     # Jabàn → Jabón
    'ö': 'í',     # Maöz → Maíz
    'µ': 'á',     # Plµtano → Plátano
    '\x90': 'é',  # Caf\x90 → Café
    '¥': 'ñ',     # Pa¥ales → Pañales
}

def fix_text(text: str) -> str:
    if not isinstance(text, str):
        return text
    for wrong, correct in CHAR_MAP.items():
        text = text.replace(wrong, correct)
    return text

def fix_df_encoding(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(fix_text)
    return df


In [50]:
df_qqp_hmo = fix_df_encoding(df_qqp_hmo)

In [51]:
df_qqp_hmo.PRODUCTO.unique()

array(['Aceite', 'Arroz', 'Atún', 'Azúcar', 'Carne Cerdo', 'Carne Pollo',
       'Carne Res', 'Cebolla', 'Chile Fresco', 'Frijol', 'Huevo',
       'Jabón de Tocador', 'Jitomate', 'Leche Ultrapasteurizada', 'Limón',
       'Manzana', 'Pan de Caja', 'Papa', 'Papel Higiénico',
       'Pasta para Sopa', 'Plátano', 'Sardina', 'Tortilla de Maíz',
       'Zanahoria', 'Celulares', 'Componentes de Audio',
       'Electrónicos de Video', 'Estufas',
       'Extractores de Jugos y Exprimidores', 'Horno de Microondas',
       'Lavadoras', 'Licuadoras', 'Pantallas', 'Planchas',
       'Refrigeradores', 'Ventiladores', 'Acondicionador y Enjuague',
       'Agua con Gas', 'Agua Sin Gas', 'Aguacate', 'Ajo', 'Alcohol',
       'Algodón', 'Alimento Preparado para Niños', 'Alimentos Preparados',
       'Alubia', 'Apio', 'Avena', 'Bebidas Hidratantes', 'Betabel',
       'Bolígrafo Tinta Negra', 'Brandy', 'Brócoli', 'Cacahuates',
       'Café Soluble', 'Café Tostado y Molido', 'Cafeteras', 'Cajeta',
       'C

In [ ]:
productos_basicos = [p for p in df_qqp['PRODUCTO'].unique() if any(word in p.lower() for word in ['carne', 'leche', 'pan', 'arroz', 'frijol', 'huevo', 'pollo', 'queso', 'tortilla', 'azúcar', 'café', 'aceite', 'harina', 'pasta', 'atún', 'sardina'])]
print('Productos de la canasta básica:')
print('\n'.join(productos_basicos[:20]))  # primeros 20 para no ser demasiado largo
print(f'\nTotal productos básicos encontrados: {len(productos_basicos)}')

# Luego, para los 5 más necesarios, basados en frecuencia
value_counts = df_qqp['PRODUCTO'].value_counts()
basic_counts = value_counts[value_counts.index.isin(productos_basicos)]
top_10 = basic_counts.head(10)
print('\nTop 10 productos básicos más frecuentes:')
print(top_10)

Productos de la canasta básica:
Pantallas
Panoto-s
Pantiprotectores
Pantoprazol
Pantozol
Pastelillos y Pan Dulce Empaquetado
Pomada de la Campana
Riopan
Aceite
Aceite de Oliva
Arroz
Carne Cerdo
Carne Pollo
Carne Res
Concentrado de Pollo
Empanizadores
Frijol
Frijoles
Harina de Arroz
Harina de Maöz

Total productos básicos encontrados: 55

Top 5 productos básicos más frecuentes:
PRODUCTO
Leche Ultrapasteurizada                6711
Carne Res                              6017
Carne Cerdo                            5812
Huevo                                  5128
Pasta para Sopa                        4946
Frijol                                 4932
Aceite                                 4666
Pastelillos y Pan Dulce Empaquetado    4584
Carne Pollo                            4259
Pan de Caja                            4256
Name: count, dtype: int64


In [61]:
# Crear un nuevo DataFrame con solo productos de la canasta básica especificada
# Corregido: usar 'carne' en lugar de 'res' para evitar falsos positivos como "Refresco"
categorias = ['leche', 'carne', 'huevo', 'frijol', 'arroz', 'maíz', 'tortilla']
df_canasta_basica = df_qqp_hmo[df_qqp_hmo['PRODUCTO'].str.contains('|'.join(categorias), case=False, na=False)]

print(f"Nuevo DataFrame creado: {df_canasta_basica.shape[0]} filas, {df_canasta_basica.shape[1]} columnas")
print("\nTop 10 productos en el nuevo DataFrame:")
print(df_canasta_basica['PRODUCTO'].value_counts().head(10))

Nuevo DataFrame creado: 575 filas, 15 columnas

Top 10 productos en el nuevo DataFrame:
PRODUCTO
Carne Cerdo                80
Carne Res                  78
Leche Ultrapasteurizada    65
Carne Pollo                56
Tortilla de Maíz           52
Arroz                      39
Frijol                     39
Huevo                      31
Leche en Polvo             29
Leche Condensada           22
Name: count, dtype: int64


In [62]:
df_canasta_basica.to_csv("df_canasta_basica.csv")